DSC540-T301 <br>
Week 5&6 Assignment<br>
Jeremy Barton <br>

### 1. Activity 5.0.1: Reading tabular data from a web page and creating Dataframes

1. Open the page in a separate Chrome/Firefox tab and use something like  `Inspect Element` tool to view the source HTML and understandn its structure.

In [1]:
# Imports and preprocessing
from bs4 import BeautifulSoup
import pandas as pd

2. Read the page using `bs4`

In [2]:
# Load in the webpage
fd = open("/home/jeremy/Documents/GitHub/DSC540/data/List of countries by GDP (nominal) - Wikipedia.html", "r", encoding = "utf-8")
# Turn variable into a soup variable
soup = BeautifulSoup(fd)
fd.close()

3. Find the table structure you will need to deal with (how many tables are there?)

In [3]:
# Locating all tables in the HTML file
all_tables = soup.find_all("table")
print("Total # of tables: {} ".format(len(all_tables)))

Total # of tables: 7 


4. Find the right table using `bs4`.

Using the `class` attribute, we can locate the correct table for our analysis.

In the class attribute for the table we need, there are FIVE classes. This is the type of hiearchical organization found on large and complex websites such as wikipedia.

`class="wikitable sortable sticky-header-multi static-row-numbers jquery-tablesorter"`

By taking only the first class, "wikitable" and the "table" element, the target table can be found.

In [4]:
data_table = soup.find("table", class_="wikitable sortable sticky-header-multi static-row-numbers jquery-tablesorter")
print(type(data_table))

<class 'bs4.element.Tag'>


5. Separate the source names and their corresponding data.

In [5]:
# Find the table
data_table = soup.find("table", class_="wikitable sortable sticky-header-multi static-row-numbers jquery-tablesorter")

# Find the header row with class "static-row-header"
header_row = data_table.find("tr", class_="static-row-header")

6. Get the source names from the list of sources you have created.

In [6]:
# Extract text from all <th> tags
headers = [th.get_text(strip=True) for th in header_row.find_all('th')]

print(headers)

['Country/Territory', 'IMF(2025)[6]', 'World Bank(2024)[7]', 'United Nations(2023)[8]']


7. Separate the header from the data that you separated before for the first source only, and then create a DataFrame using that.

In [7]:
# Extract all data rows (excluding the header row with class "static-row-header")
all_rows = data_table.find('tbody').find_all('tr')
data_rows = [row for row in all_rows if 'static-row-header' not in row.get('class', [])]

# Extract data from each row for IMF (first source - column index 1)
countries = []
imf_data = []

In [8]:
for row in data_rows:
    cells = row.find_all(['td', 'th'])
    if len(cells) >= 2:  # Make sure we have country and IMF columns
        # Extract country name
        country = cells[0].get_text(strip=True)
        # Extract IMF GDP value (column index 1)
        imf = cells[1].get_text(strip=True)
        
        countries.append(country)
        imf_data.append(imf)

In [9]:
# Create DataFrame for IMF data
df_imf = pd.DataFrame({
    'Country/Territory': countries,
    'GDP(US$MM)': imf_data
})

print("IMF (2025) DataFrame:")
print(df_imf.head())
print(f"\nShape: {df_imf.shape}")

IMF (2025) DataFrame:
  Country/Territory  GDP(US$MM)
0     United States  30,615,743
1        China[n 1]  19,398,577
2           Germany   5,013,574
3             Japan   4,279,828
4             India   4,125,213

Shape: (212, 2)


8. Repeat the last task for the other two data sources.

In [10]:
# Extract data for World Bank (column index 2)
world_bank_data = []

for row in data_rows:
    cells = row.find_all(['td', 'th'])
    if len(cells) >= 3:  # Make sure we have country and World Bank columns
        # Extract World Bank GDP value (column index 2)
        wb = cells[2].get_text(strip=True)
        world_bank_data.append(wb)

# Create DataFrame for World Bank data
df_world_bank = pd.DataFrame({
    'Country/Territory': countries,
    'GDP(US$MM)': world_bank_data
})

print("=" * 40)
print("World Bank (2024) DataFrame:")
print("=" * 40)
print(df_world_bank.head())
print(f"\nShape: {df_world_bank.shape}")
print()

# Extract data for United Nations (column index 3)
un_data = []

for row in data_rows:
    cells = row.find_all(['td', 'th'])
    if len(cells) >= 4:  # Make sure we have country and UN columns
        # Extract UN GDP value (column index 3)
        un = cells[3].get_text(strip=True)
        un_data.append(un)

# Create DataFrame for United Nations data
df_un = pd.DataFrame({
    'Country/Territory': countries,
    'GDP(US$MM)': un_data
})

print("=" * 40)
print("United Nations (2023) DataFrame:")
print("=" * 40)
print(df_un.head())
print(f"\nShape: {df_un.shape}")

World Bank (2024) DataFrame:
  Country/Territory  GDP(US$MM)
0     United States  28,750,956
1        China[n 1]  18,743,803
2           Germany   4,685,593
3             Japan   4,027,598
4             India   3,909,892

Shape: (212, 2)

United Nations (2023) DataFrame:
  Country/Territory  GDP(US$MM)
0     United States  27,720,700
1        China[n 1]  17,794,782
2           Germany   4,525,704
3             Japan   4,204,495
4             India   3,575,778

Shape: (212, 2)


### 1. Activity 5.0.1: Reading tabular data from a web page and creating Dataframes

1. Read in `visit_data.csv` file.

In [11]:
visit_data = pd.read_csv('/home/jeremy/Documents/GitHub/DSC540/data/visit_data.csv')

2. Check for duplicates

The duplicated() method returns a Series with True of false valuses that describe which rows in the DataFrame are duplicated, and not.

GeeksforGeeks. “Pandas DataFrame Duplicated() Method Python.” GeeksforGeeks, 23 July 2018, www.geeksforgeeks.org/python/pandas-dataframe-duplicated/.

In [12]:
print("Duplicates for First name: {}"\
.format(any(visit_data.first_name.duplicated())))
print("Duplicates for Last name: {}"\
.format(any(visit_data.last_name.duplicated())))
print("Duplicates for Email: {}"\
.format(any(visit_data.email.duplicated())))

Duplicates for First name: True
Duplicates for Last name: True
Duplicates for Email: False


3. Check whether any essential column contains NaN

Using the % operator to find remainders, NaNs can be located by pairing the .isnull() with the .any() method. This looks for Nulls in any fields in the datasets.

In [13]:
print("The column Email contains NaN - %r " % \
visit_data.email.isnull().values.any())
print("The column IP Address contains NaN - %s " % \
visit_data.ip_address.isnull().values.any())
print("The column Visit contains NaN - %s " % \
visit_data.visit.isnull().values.any())

The column Email contains NaN - np.False_ 
The column IP Address contains NaN - False 
The column Visit contains NaN - True 


4. Get rid of outliers

To find outliers, we'll be using the z score. Z scores measuren how many standard deviations a data point is from the mean. It basically measures this as a distance with respect to the variability of the entire dataset.

In [14]:
from scipy import stats
import numpy as np

# Calculate z-scores for numerical 'visit' column
z_scores = np.abs(stats.zscore(visit_data['visit']))

# Identify outliers (z-score > 3)
outliers = visit_data[z_scores > 3]
print(f"Found {len(outliers)} outliers")

# Remove outliers
visit_data_clean = visit_data[z_scores <= 3]

Found 0 outliers


This method didn't find any outliers, so let's use the books answer.

In [15]:
size_prev = visit_data.shape
visit_data = visit_data[np.isfinite(visit_data['visit'])]
#This is an inplace operation.
# After this operation the original DataFrame is lost.
size_after = visit_data.shape

Using .isinfinite() is a data cleaning approach looking for infinite values, NaNs and negative infinite values. It retains finite numerical values.

In [16]:
print("The size of previous data was - {prev[0]} rows and "\
"the size of the new one is - {after[0]} rows"\
.format(prev=size_prev, after=size_after))

The size of previous data was - 1000 rows and the size of the new one is - 974 rows


3. Insert data into a SQL Lite database – create a table with the following data below that you will create yourself (Hint on how to create the SQL: Python for Data Analysis 2nd edition page 191, Python for Data Analysis 3rd Edition: Page 199):


     a. Name, Address, City, State, Zip, Phone Number

     b. Add at least 10 rows of data and submit your code with a query generating your results.

In [17]:
# Import sqlite3
import sqlite3

In [18]:
# Building the query
query = """
CREATE TABLE visit_table
(
  NAME TEXT, 
  ADDRESS TEXT,
  CITY TEXT, 
  STATE TEXT,
  ZIP TEXT,
  PHONE TEXT
); """

drop_table = """
DROP TABLE visit_table;
"""

It may be first instinct to use `INTEGER` for the PHONE and ZIP field due to it appearing numeric, but these fields will not be used for calulations in processing. The `TEXT` datatype is more fitting for them, especially for the case of leading zeroes.

Now, creating a sqlite connection.

In [21]:
con = sqlite3.connect("mydata.sqlite")
con.execute(query)
con.commit()

Creating at least 10 rows of data.

Rows were computer generated to mimic a larger dataset.

In [22]:
data = [
    ("John Smith", "123 Main St", "New York", "NY", "10001", "(212) 555-0101"),
    ("Maria Garcia", "456 Oak Ave", "Los Angeles", "CA", "90001", "(310) 555-0102"),
    ("James Johnson", "789 Pine Rd", "Chicago", "IL", "60601", "(312) 555-0103"),
    ("Linda Williams", "321 Elm St", "Houston", "TX", "77001", "(713) 555-0104"),
    ("Robert Brown", "654 Maple Dr", "Phoenix", "AZ", "85001", "(602) 555-0105"),
    ("Patricia Jones", "987 Cedar Ln", "Philadelphia", "PA", "19019", "(215) 555-0106"),
    ("Michael Davis", "147 Birch Way", "San Antonio", "TX", "78201", "(210) 555-0107"),
    ("Jennifer Miller", "258 Spruce Ct", "San Diego", "CA", "92101", "(619) 555-0108"),
    ("William Wilson", "369 Ash Blvd", "Dallas", "TX", "75201", "(214) 555-0109"),
    ("Elizabeth Moore", "741 Willow Pl", "San Jose", "CA", "95101", "(408) 555-0110"),
    ("David Taylor", "852 Cherry St", "Austin", "TX", "78701", "(512) 555-0111"),
    ("Barbara Anderson", "963 Poplar Ave", "Jacksonville", "FL", "32099", "(904) 555-0112"),
    ("Richard Thomas", "159 Hickory Rd", "Fort Worth", "TX", "76101", "(817) 555-0113"),
    ("Susan Jackson", "267 Walnut Dr", "Columbus", "OH", "43004", "(614) 555-0114"),
    ("Joseph White", "348 Dogwood Ln", "Charlotte", "NC", "28201", "(704) 555-0115"),
    ("Jessica Harris", "426 Magnolia Way", "San Francisco", "CA", "94102", "(415) 555-0116"),
    ("Thomas Martin", "537 Sycamore Ct", "Indianapolis", "IN", "46201", "(317) 555-0117"),
    ("Sarah Thompson", "648 Redwood Blvd", "Seattle", "WA", "98101", "(206) 555-0118"),
    ("Charles Garcia", "759 Cypress Pl", "Denver", "CO", "80201", "(303) 555-0119"),
    ("Nancy Martinez", "861 Fir St", "Washington", "DC", "20001", "(202) 555-0120"),
    ("Daniel Robinson", "972 Hemlock Ave", "Boston", "MA", "02101", "(617) 555-0121"),
    ("Lisa Clark", "183 Juniper Rd", "Nashville", "TN", "37201", "(615) 555-0122"),
    ("Matthew Rodriguez", "294 Locust Dr", "El Paso", "TX", "79901", "(915) 555-0123"),
    ("Karen Lewis", "315 Beech Ln", "Detroit", "MI", "48201", "(313) 555-0124"),
    ("Donald Lee", "426 Chestnut Way", "Portland", "OR", "97201", "(503) 555-0125"),
    ("Betty Walker", "537 Alder Ct", "Las Vegas", "NV", "89101", "(702) 555-0126"),
    ("Paul Hall", "648 Cottonwood Blvd", "Oklahoma City", "OK", "73101", "(405) 555-0127"),
    ("Dorothy Allen", "759 Basswood Pl", "Memphis", "TN", "38101", "(901) 555-0128"),
    ("Mark Young", "861 Hawthorn St", "Louisville", "KY", "40201", "(502) 555-0129"),
    ("Sandra Hernandez", "972 Ironwood Ave", "Baltimore", "MD", "21201", "(410) 555-0130"),
    ("Steven King", "183 Linden Rd", "Milwaukee", "WI", "53201", "(414) 555-0131"),
    ("Ashley Wright", "294 Mesquite Dr", "Albuquerque", "NM", "87101", "(505) 555-0132"),
    ("Kenneth Lopez", "315 Olive Ln", "Tucson", "AZ", "85701", "(520) 555-0133"),
    ("Donna Hill", "426 Palm Way", "Fresno", "CA", "93650", "(559) 555-0134"),
    ("Joshua Scott", "537 Pecan Ct", "Sacramento", "CA", "94203", "(916) 555-0135"),
    ("Carol Green", "648 Sequoia Blvd", "Kansas City", "MO", "64101", "(816) 555-0136"),
    ("Kevin Adams", "759 Teak Pl", "Mesa", "AZ", "85201", "(480) 555-0137"),
    ("Michelle Baker", "861 Tulip St", "Atlanta", "GA", "30301", "(404) 555-0138"),
    ("Brian Nelson", "972 Violet Ave", "Omaha", "NE", "68101", "(402) 555-0139"),
    ("Emily Carter", "183 Acacia Rd", "Colorado Springs", "CO", "80903", "(719) 555-0140"),
    ("George Mitchell", "294 Bamboo Dr", "Raleigh", "NC", "27601", "(919) 555-0141"),
    ("Amanda Perez", "315 Cactus Ln", "Miami", "FL", "33101", "(305) 555-0142"),
    ("Ronald Roberts", "426 Dahlia Way", "Long Beach", "CA", "90801", "(562) 555-0143"),
    ("Deborah Turner", "537 Eucalyptus Ct", "Virginia Beach", "VA", "23450", "(757) 555-0144"),
    ("Jason Phillips", "648 Gardenia Blvd", "Oakland", "CA", "94601", "(510) 555-0145"),
    ("Stephanie Campbell", "759 Hyacinth Pl", "Minneapolis", "MN", "55401", "(612) 555-0146"),
    ("Ryan Parker", "861 Iris St", "Tulsa", "OK", "74101", "(918) 555-0147"),
    ("Rebecca Evans", "972 Jasmine Ave", "Wichita", "KS", "67201", "(316) 555-0148"),
    ("Gary Edwards", "183 Lavender Rd", "Arlington", "TX", "76010", "(817) 555-0149"),
    ("Laura Collins", "294 Marigold Dr", "New Orleans", "LA", "70112", "(504) 555-0150")
]

Insert the data.

In [23]:
# Building the INSERT statement
stmt = "INSERT INTO visit_table VALUES(?, ?, ?, ?, ?, ?)"
con.executemany(stmt, data)
con.commit()

SELECT the data to see the results. 

In [24]:
cursor = con.execute("""
        SELECT NAME, ADDRESS, CITY, STATE, ZIP, PHONE
            FROM visit_table 
            GROUP BY STATE, NAME
            ;
        """)

In [25]:
rows = cursor.fetchall()

# Extract column names from cursor.description
#   Using the tuple option to add the data
#   and combining the output below is useful for very large datasets.
columns = [x[0] for x in cursor.description]

# Create dataframe from columns and rows
visit_df = pd.DataFrame(rows, columns=columns)

print(visit_df)


                  NAME              ADDRESS              CITY STATE    ZIP  \
0        Kenneth Lopez         315 Olive Ln            Tucson    AZ  85701   
1          Kevin Adams          759 Teak Pl              Mesa    AZ  85201   
2         Robert Brown         654 Maple Dr           Phoenix    AZ  85001   
3           Donna Hill         426 Palm Way            Fresno    CA  93650   
4      Elizabeth Moore        741 Willow Pl          San Jose    CA  95101   
5       Jason Phillips    648 Gardenia Blvd           Oakland    CA  94601   
6      Jennifer Miller        258 Spruce Ct         San Diego    CA  92101   
7       Jessica Harris     426 Magnolia Way     San Francisco    CA  94102   
8         Joshua Scott         537 Pecan Ct        Sacramento    CA  94203   
9         Maria Garcia          456 Oak Ave       Los Angeles    CA  90001   
10      Ronald Roberts       426 Dahlia Way        Long Beach    CA  90801   
11      Charles Garcia       759 Cypress Pl            Denver   